In [ ]:
!sudo apt-get install -y pciutils lshw zstd
!curl -fsSL https://ollama.com/install.sh | sh
! sudo systemctl restart ollama
! ollama --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 100 not upgraded.
Need to get 1,486 kB of archives.
After this operation, 4,951 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy/main

In [ ]:
import subprocess, time, os, requests

  # Kill existing ollama serve
subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True)
time.sleep(2)

  # Restart with origins open
env = os.environ.copy()
env['OLLAMA_ORIGINS'] = '*'

proc = subprocess.Popen(
      ['ollama', 'serve'],
      stdout=subprocess.DEVNULL,
      stderr=subprocess.DEVNULL,
      env=env
)
time.sleep(3)

  # Verify
r = requests.get("http://localhost:11434/api/tags")
print("✓ Ollama running:", r.status_code)

✓ Ollama running: 200


In [ ]:
!ollama pull qwen3.5:9b

In [ ]:
!ollama list

NAME          ID              SIZE      MODIFIED               
qwen3.5:9b    6488c96fa5fa    6.6 GB    Less than a second ago    


Error: model 'qwen2.5:7b' not found


In [ ]:
import requests, json
r = requests.post("http://localhost:11434/api/generate",
      json={"model": "qwen3.5:9b", "prompt": "hi", "stream": False})
print(r.json())

{'model': 'qwen3.5:9b', 'created_at': '2026-05-11T11:05:09.186478276Z', 'response': 'Hello! How can I help you today?', 'thinking': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: "hi"\n    *   Intent: Greeting.\n    *   Expected Response: A friendly, polite greeting in return, offering assistance.\n\n2.  **Determine the Tone:**\n    *   Casual, friendly, helpful.\n\n3.  **Draft Possible Responses:**\n    *   "Hello! How can I help you today?"\n    *   "Hi there! What\'s on your mind?"\n    *   "Hey! How\'s it going?"\n\n4.  **Select the Best Response:**\n    *   Keep it simple and open-ended to encourage further interaction.\n    *   "Hello! How can I help you today?" is standard and effective.\n    *   Let\'s make it slightly warm: "Hi there! How\'s your day going? Is there anything I can help you with?"\n\n5.  **Final Polish:**\n    *   "Hello! How can I help you today?" (Simple and direct)\n    *   Or just "Hi there! How can I help you?"\n\n    Let\'s go with a f

In [ ]:
import subprocess
subprocess.run(['fuser', '-k', '11435/tcp'], capture_output=True)


CompletedProcess(args=['fuser', '-k', '11435/tcp'], returncode=1, stdout=b'', stderr=b'')

In [ ]:
import threading
from http.server import HTTPServer, BaseHTTPRequestHandler
import urllib.request

class OllamaProxy(BaseHTTPRequestHandler):
      def do_request(self):
          length = int(self.headers.get('Content-Length', 0))
          body = self.rfile.read(length) if length > 0 else None
          headers = {k: v for k, v in self.headers.items()
                     if k.lower() not in ('host', 'content-length')}
          if body:
              headers['Content-Length'] = str(len(body))
          try:
              req = urllib.request.Request(
                  f'http://localhost:11434{self.path}',
                  data=body, headers=headers, method=self.command
              )
              with urllib.request.urlopen(req, timeout=300) as resp:
                  self.send_response(resp.status)
                  for k, v in resp.headers.items():
                      if k.lower() != 'transfer-encoding':
                          self.send_header(k, v)
                  self.end_headers()
                  while chunk := resp.read(4096):
                      self.wfile.write(chunk)
          except Exception as e:
              self.send_error(502, str(e))

      do_GET = do_POST = do_PUT = do_DELETE = do_request
      def log_message(self, *_): pass

server = HTTPServer(('0.0.0.0', 11435), OllamaProxy)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("✓ Proxy on :11435")

OSError: [Errno 98] Address already in use

In [ ]:
! curl -L https://github.com/ekzhang/bore/releases/download/v0.5.0/bore-v0.5.0-x86_64-unknown-linux-musl.tar.gz -o /tmp/bore.tar.gz && tar -xzf /tmp/bore.tar.gz -C /usr/local/bin && chmod +x /usr/local/bin/bore && bore --version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2007k  100 2007k    0     0  4309k      0 --:--:-- --:--:-- --:--:-- 4309k
bore-cli 0.5.0


In [ ]:
import subprocess, threading, time, re

proc = subprocess.Popen(
      ['bore', 'local', '11435', '--to', 'bore.pub'],
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
  )

BORE_URL = None
def read():
      global BORE_URL
      for line in proc.stdout:
          print(line, end='')
          m = re.search(r'bore\.pub:(\d+)', line)
          if m and not BORE_URL:
              BORE_URL = f'http://bore.pub:{m.group(1)}'
              print(f'\n✓ Tunnel: {BORE_URL}')

threading.Thread(target=read, daemon=False).start()
time.sleep(10)
print('BORE_URL =', BORE_URL)

2026-05-11T12:58:34.376686Z  INFO bore_cli::client: connected to server remote_port=8526
2026-05-11T12:58:34.376814Z  INFO bore_cli::client: listening at bore.pub:8526

✓ Tunnel: http://bore.pub:8526
BORE_URL = http://bore.pub:8526


In [ ]:
import requests, subprocess, re

  # Check Ollama
try:
      r = requests.get("http://localhost:11434/api/tags", timeout=5)
      print("✓ Ollama alive:", r.status_code)
except Exception as e:
      print("✗ Ollama down:", e)

  # Check proxy on :11435
try:
      r = requests.get("http://localhost:11435/api/tags", timeout=5)
      print("✓ Proxy on :11435 alive:", r.status_code)
except Exception as e:
      print("✗ Proxy on :11435 down:", e)

  # Check bore process + extract port
result = subprocess.run(['pgrep', '-a', '-f', 'bore local'], capture_output=True, text=True)
if result.stdout.strip():
      print("✓ bore running:", result.stdout.strip())
      m = re.search(r'bore local (\d+)', result.stdout)
      port = m.group(1) if m else "unknown"
      print(f"  Listening on: bore.pub (local port {port})")
else:
      print("✗ bore NOT running")

  # Verify bore tunnel reachable — update port below
# BORE_URL = "http://bore.pub:27425"  # ← update this each session
try:
      r2 = requests.get(f"{BORE_URL}/api/tags", timeout=10)
      print("✓ Bore tunnel reachable:", r2.status_code, BORE_URL)
except Exception as e:
      print("✗ Bore tunnel unreachable:", e)

  # Check OLLAMA_ORIGINS
result = subprocess.run(['pgrep', '-f', 'ollama serve'], capture_output=True, text=True)
pid = result.stdout.strip().split('\n')[0]
if pid:
      env_bytes = open(f'/proc/{pid}/environ', 'rb').read()
      env_str = env_bytes.replace(b'\x00', b'\n').decode(errors='ignore')
      print("OLLAMA_ORIGINS set:", 'OLLAMA_ORIGINS' in env_str)
      print("Value:", [l for l in env_str.split('\n') if 'OLLAMA_ORIGINS' in l])

✓ Ollama alive: 200
✓ Proxy on :11435 alive: 200
✓ bore running: 7946 bore local 11435 --to bore.pub
  Listening on: bore.pub (local port 11435)
2026-05-11T12:58:44.396546Z  INFO proxy{id=51a66cac-a572-42af-bba7-df4f9932198a}: bore_cli::client: new connection
2026-05-11T12:58:44.535307Z  INFO proxy{id=51a66cac-a572-42af-bba7-df4f9932198a}: bore_cli::client: connection exited
✓ Bore tunnel reachable: 200 http://bore.pub:8526
OLLAMA_ORIGINS set: True
Value: ['OLLAMA_ORIGINS=*']


In [ ]:
import time, requests
print("Keep-alive running — leave this cell running")
while True:
      try: requests.get("http://localhost:11434/api/tags")
      except: pass
      time.sleep(30)

Keep-alive running — leave this cell running
2026-05-11T10:56:28.270704Z  INFO proxy{id=08e06b0c-1b6b-4029-9a37-f74b7ffe6f47}: bore_cli::client: new connection
2026-05-11T10:56:28.815268Z  INFO proxy{id=08e06b0c-1b6b-4029-9a37-f74b7ffe6f47}: bore_cli::client: connection exited
